In [ ]:
# Installing packages and adding colormaps
import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from matplotlib import colors, patches
import matplotlib.image as mpimg
import zarr
import random
import sys
import zipfile
import io
import json
import os
import pickle

# Colormaps for both datasets
colors_phiSat = ['green', 'red', 'lightgray', 'blue']
my_cmap_phiSat = ListedColormap(colors_phiSat)

box_0_phiSat = mpatches.Patch(color=colors_phiSat[0], label='0: Background')
box_1_phiSat = mpatches.Patch(color=colors_phiSat[1], label='1: Burned area')
box_2_phiSat = mpatches.Patch(color=colors_phiSat[2], label='2: Clouds')
box_3_phiSat = mpatches.Patch(color=colors_phiSat[3], label='3: Water')

colors_THRawS = ['black', 'orange', 'red']
my_cmap_THRawS = ListedColormap(colors_THRawS)

box_0_THRawS = mpatches.Patch(color=colors_THRawS[0], label='0: Certainly not burned area')
box_1_THRawS = mpatches.Patch(color=colors_THRawS[1], label='1: Maybe burned area')
box_2_THRawS = mpatches.Patch(color=colors_THRawS[2], label='2: No burned area')

In [ ]:
# PhiSat2 dataset
path_phiSat2 = "/home/laura/Scriptie/ruwe_data/burned.zarr.zip"
store_phiSat2 = zarr.ZipStore(path_phiSat2, mode='r')

In [ ]:
# SegTHRawS dataset
path_SegTHRawS = "/home/laura/Scriptie/ruwe_data/Data_Hotspots.zip"
store_SegTHRawS = zarr.ZipStore(path_SegTHRawS, mode='r')

In [ ]:
# THRawS dataset
path_THRawS = "/home/laura/Scriptie/data/THRawS.zip"

try:
    with zipfile.ZipFile(path_THRawS, "r") as zip_ref:
        with zip_ref.open('L1C_files.json') as file:
            data = json.load(file)

    all_keys = list(data.keys())
    print(f"Total number of found items in JSON: {len(all_keys)}\n")

    event_list = ['Barren_Island', 'Chillan_Nevados_de', 'Etna', 'Fuego', 'Kilauea', 'Krakatau', 'Piton_de_la_Fournaise', 'Stromboli', 'Taal'] 

    NE = []
    eruption = []
    wildfire = []

    for name in all_keys:
        if '_NE_' in name:
            NE.append(name)
        elif any(vulcano in name for vulcano in event_list):
            eruption.append(name)
        else:
            wildfire.append(name)

        all_files = zip_ref.namelist()
    for f in all_files[:15]: 
        print(f" - {f}")

    brand_key = next((k for k in all_keys if 'wildfire' in k.lower() or 'fire' in k.lower() or 'australia' in k.lower()), all_keys[10])
    ne_key = next((k for k in all_keys if '_NE_' in k), all_keys[0])

    print(f"Brand-label ('{brand_key}'):")
    print(json.dumps(data[brand_key], indent=4))
        
    print(f"\nNo-Event label ('{ne_key}'):")
    print(json.dumps(data[ne_key], indent=4))

    #print(" --- Distribution in THRawS ---")
    #print(f"No Event: {len(NE)}")
    #print(f"Eruption: {len(eruption)}")
    #print(f"Wildfire: {len(wildfire)}")

except Exception as e:
    print(f"Error opening THRawS JSON: {e}")

In [ ]:
# phiSat2 dataset
raw_data_dir = '/home/laura/Scriptie/ruwe_data'
path_phiSat2 = os.path.join(raw_data_dir, 'burned.zarr.zip')

try:
    store_phiSat2 = zarr.ZipStore(path_phiSat2, mode='r')
    zarr_root = zarr.group(store=store_phiSat2)
    
    for name, item in zarr_root['trainval'].items():
        print(f" - {name} (Type: {type(item)})")
        
    store_phiSat2.close()
    
except Exception as e:
    print(f"Error: {e}")



In [ ]:
# phiSat2 dataset
path_phiSat2 = "/home/laura/Scriptie/ruwe_data/burned.zarr.zip"
store_phiSat2 = zarr.ZipStore(path_phiSat2, mode='r')

all_keys = list(store_phiSat2.keys())
samples = list(set([key.split('/')[1] for key in all_keys if key.startswith('trainval/')]))
test_sample = "0007783"

image = np.array(zarr.open_array(store_phiSat2, path=f"trainval/{test_sample}/img", mode='r'))
label = np.array(zarr.open_array(store_phiSat2, path=f"trainval/{test_sample}/label", mode='r'))

nir = image[3]
swir = image[6]

nbr = (nir - swir) / (nir + swir + 1e-8 )  # Adding a small constant to avoid division by zero

if len(label.shape) == 3 and label.shape[0] > 1: #Waarom die 2e?
    layered_label = np.argmax(label, axis=0)
else:
    layered_label = label

def strech_rgb(band):
    p2, p98 = np.percentile(band, (2, 98))
    return np.clip((band - p2) / (p98 - p2), 0, 1)

red = strech_rgb(image[2]. astype(np.float32))
green = strech_rgb(image[1]. astype(np.float32))
blue = strech_rgb(image[0]. astype(np.float32))

rgb_image = np.dstack((red, green, blue))

plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1)
plt.imshow(nbr, cmap='viridis')
plt.title('NBR Index')
plt.colorbar(label = "NBR value")

plt.subplot(1, 3, 2)
plt.imshow(layered_label, cmap=my_cmap_phiSat, vmin=0, vmax=len(colors_phiSat)-1)
plt.title('Label')
plt.legend(handles=[box_0_phiSat, box_1_phiSat, box_2_phiSat, box_3_phiSat], 
           bbox_to_anchor=(1.05, 1), loc='upper left')

plt.subplot(1, 3, 3)
plt.imshow(rgb_image)
plt.title('RGB Image')
plt.axis('off')

plt.tight_layout()
plt.show()

store_phiSat2.close()

In [ ]:
# SegTHRawS dataset
path_SegTHRawS = "/home/laura/Scriptie/ruwe_data/Data_Hotspots.zip"
img_target = 'event/NIR_SWIR/Etna_02_G0_(192, 576, 448, 832)_NIR_SWIR.pkl'
mask_target = 'Etna_02_G0_(192, 576, 448, 832)'

try:
    with zipfile.ZipFile(path_SegTHRawS, 'r') as outer_zip:
        with outer_zip.open('SegTHRawS_images_event.zip') as inner_images_file:
            with zipfile.ZipFile(inner_images_file) as inner_images_zip:
                with inner_images_zip.open(img_target) as f:
                    img_data = pickle.load(f)
                    
        with outer_zip.open('SegTHRawS_masks.zip') as inner_masks_file:
            with zipfile.ZipFile(inner_masks_file) as inner_masks_zip:
                all_masks = inner_masks_zip.namelist()
                mask_target = next(m for m in all_masks if mask_target in m)
                
                with inner_masks_zip.open(mask_target) as f:
                    mask_data = pickle.load(f)

    print(f"\n Image type: {type(img_data)}")
    if isinstance(img_data, np.ndarray):
        print(f"\n Image shape: {img_data.shape}")
        
    print(f"\n Mask type: {type(mask_data)}")
    if isinstance(mask_data, np.ndarray):
        print(f"\n Mask shape: {mask_data.shape}")

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    if len(img_data.shape) == 3:
        plot_img = img_data[0] if img_data.shape[0] < img_data.shape[2] else img_data[:,:,0]
    else:
        plot_img = img_data
    plt.imshow(plot_img, cmap='gray')
    plt.title(f'Image: {img_data.shape}')

    plt.subplot(1, 2, 2)
    if len(mask_data.shape) == 3:
        plot_mask = mask_data[0] if mask_data.shape[0] < mask_data.shape[2] else mask_data[:,:,0]
    else:
        plot_mask = mask_data
    plt.imshow(plot_mask, cmap=my_cmap_THRawS, vmin=0, vmax=2)
    plt.title(f'Mask: {mask_data.shape}')
    plt.legend(handles=[box_0_THRawS, box_1_THRawS, box_2_THRawS], 
               bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error: {e}")